# Report reconstruction figures

Exports matched saved prior and low-level outputs into the report-local images directory. Reads CSV/YAML provenance and saved PNGs; does not regenerate predictions or launch training. Test image ordering matches get_image_paths in src/brain_image/data/io.py.


In [1]:
from pathlib import Path
import os
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from PIL import Image

repo = Path.cwd().resolve()
if repo.name == 'notebooks':
    repo = repo.parent
os.chdir(repo)
images_dir = repo / 'report/69bd47171c8bd99594e793c6/images'
images_dir.mkdir(exist_ok=True)
prior_runs = {
    'Prior-only': repo / 'experiments/eeg_prior/20260831_134809/train_eeg-20260831_134844/version_0',
    'Joint alignment + prior': repo / 'experiments/eeg_alignprior/20260904_100813/train_eeg-20260904_100911/version_0',
}
low_runs = {
    'MSE': repo / 'experiments/low_level/20260905_025455/train_low_level-20260905_025524/version_0',
    'Unaligned SynCLR': repo / 'experiments/low_level/unaligned_cached_20260905/train_low_level-20260905_180009/version_0',
    'Aligned SynCLR': repo / 'experiments/low_level/aligned_cached_20260906/train_low_level-20260906_140857/version_0',
}
for name, run in {**prior_runs, **low_runs}.items():
    metrics = pd.read_csv(run / 'test/test_metrics.csv').set_index('metric')['value']
    evaluation = yaml.safe_load((run / 'test/evaluation_config.yaml').read_text())
    checkpoint = repo / evaluation['checkpoint_path']
    assert checkpoint.exists(), checkpoint
    if name in low_runs:
        assert evaluation['dataset']['subs'] == [8]
        assert evaluation['dataset']['test_batch_size'] == 1
    else:
        hparams = yaml.safe_load((run / 'hparams.yaml').read_text())
        assert hparams['dataset_config']['subs'] == [8]
        assert len(list((run / 'test').glob('*/test/prior/recon.png'))) == 9
    print(name, checkpoint.name)
    display(metrics.to_frame(name))


Prior-only epoch_0289-eval_val_prior_pred_cos_0.6340.ckpt


,Prior-only
metric,
eval/test/prior/pred_cos,0.633960
eval/test/prior/recon/pixcorr,0.097220
eval/test/prior/recon/ssim,0.308554
eval/test/prior/recon/alex2,0.722222
eval/test/prior/recon/alex5,0.888889
eval/test/prior/recon/inceptionv3,0.638889
eval/test/prior/recon/clip,0.791667
eval/test/prior/recon/efficientnet,0.900438
eval/test/prior/recon/swav,0.587954


Joint alignment + prior epoch_0409-eval_val_prior_pred_cos_0.6420.ckpt


,Joint alignment + prior
metric,
eval/test/align/brain_acc,0.595000
eval/test/align/image_acc,0.515000
eval/test/align/sim,0.264222
eval/test/prior/pred_cos,0.641994
eval/test/prior/recon/pixcorr,0.181082
eval/test/prior/recon/ssim,0.315726
eval/test/prior/recon/alex2,0.944444
eval/test/prior/recon/alex5,0.916667
eval/test/prior/recon/inceptionv3,0.805556


MSE epoch_0159-val_loss_0.6073.ckpt


,MSE
metric,
test/loss,0.607279
test/losses/l2_loss,0.607279
test/pixcorr,0.243123
test/ssim,0.366912
test/alex2,0.701608
test/alex5,0.696457
test/inceptionv3,0.547889
test/clip,0.516156
test/efficientnet,0.973867


Unaligned SynCLR epoch_0089-val_loss_0.8114.ckpt


,Unaligned SynCLR
metric,
test/loss,0.811388
test/losses/l2_loss,0.624766
test/losses/perceptual_loss,0.186622
test/pixcorr,0.197601
test/ssim,0.417105
test/alex2,0.731558
test/alex5,0.711709
test/inceptionv3,0.542513
test/clip,0.552462


Aligned SynCLR epoch_0069-val_loss_0.9002.ckpt


,Aligned SynCLR
metric,
test/loss,0.900243
test/losses/l2_loss,0.629270
test/losses/perceptual_loss,0.270973
test/pixcorr,0.185913
test/ssim,0.408515
test/alex2,0.790502
test/alex5,0.731457
test/inceptionv3,0.561357
test/clip,0.538090


In [2]:
examples = ['jelly_bean_06s', 'scallop_02s', 'wine_05s']
fig, axes = plt.subplots(3, 3, figsize=(8, 8.2))
for row, example in enumerate(examples):
    directories = [run / 'test' / f'{example}_eval/test/prior' for run in prior_runs.values()]
    with Image.open(directories[0] / 'gt.png') as first, Image.open(directories[1] / 'gt.png') as second:
        assert first.size == second.size
        assert first.tobytes() == second.tobytes(), example
    paths = [directories[0] / 'gt.png'] + [directory / 'recon.png' for directory in directories]
    for col, path in enumerate(paths):
        axes[row, col].imshow(Image.open(path).convert('RGB'))
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(['Ground truth', *prior_runs][col], fontsize=11)
fig.tight_layout()
fig.savefig(images_dir / 'prior_comparison_grid.png', dpi=240, bbox_inches='tight')
plt.show()


/home/gasparyanartur/dev/brain-image-implementation/tmp/ipykernel_947452/1143891828.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
test_dir = repo / 'data/things-eeg2/imgs/test_images'
test_paths = [path for directory in sorted(test_dir.iterdir()) for path in sorted(directory.iterdir()) if path.is_file() and path.suffix == '.jpg']
assert len(test_paths) == 200
positions = [0, 50, 100, 150, 199]
fig, axes = plt.subplots(5, 4, figsize=(8.6, 10.8))
for row, index in enumerate(positions):
    paths = [test_paths[index]] + [run / f'test/reconstructions/example_{index:03d}.png' for run in low_runs.values()]
    for col, path in enumerate(paths):
        assert path.exists(), path
        axes[row, col].imshow(Image.open(path).convert('RGB'))
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(['Ground truth', *low_runs][col], fontsize=10)
    print(index, test_paths[index].stem)
fig.tight_layout()
fig.savefig(images_dir / 'low_level_comparison_grid.png', dpi=240, bbox_inches='tight')
plt.show()


0 aircraft_carrier_06s
50 cordon_bleu_08s
100 jelly_bean_06s
150 robot_02s
199 wok_12s


/home/gasparyanartur/dev/brain-image-implementation/tmp/ipykernel_947452/24440328.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
